In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch import distributions as dist 
import numpy as np
from engression.loss_func import energy_loss, energy_loss_two_sample

from sem2 import SEM2
from nn_models import MLP, MLPConfig

In [37]:
def generate_true_data(n_samples, beta=None):
    u1 = dist.Exponential(1.0).sample((n_samples,1)) 
    u2 = dist.Uniform(low=0.0, high=1.0).sample((n_samples,1))
    u3=dist.Normal(loc=0.0, scale=1.0).sample((n_samples,1))
    x = torch.sin(u1) +4*u2**3-u2**2  
    y = x*beta[0] + u3*beta[1]+1 
    return torch.cat([u1, u2,u3, x, y], dim=1)

model_formula = 'u1 ~ Exponential(1.0); u2 ~ Uniform(0,1);u3 ~ Normal(0,1); x <- n(u1,u2); y <- x +u3'
config = MLPConfig(
    input_dim=2,
    hidden_dims=[64, 64],
    output_dim=1,
    prob_output=False,
    dropout=0.0,
    activation="relu"
)
model_components={
    "x": config
}

model = SEM2(model_formula=model_formula, mlp_cfgs=model_components)

beta_true=[2,1]
dataset=generate_true_data(3000,beta=beta_true)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
model.train()
niter = 8000
for i in range(niter):
    optimizer.zero_grad()
    sample = model(dataset.size(0) * 2)
    loss, s1, s2 = energy_loss(dataset, sample)
    loss.backward()
    optimizer.step()
    if i == 0 or (i + 1) % 1000 == 0:
        print(f'Iter {i + 1} | loss: {loss.item():.4f}, s1: {s1.item():.4f}, s2: {s2.item():.4f}')

model.eval()
with torch.no_grad():
        beta_hat = model.layer['y'].weight_matrix()[0].detach().cpu().numpy().flatten()
        bias_hat = model.layer['y'].weight_matrix()[1].detach().cpu().numpy().flatten()
print(f"\nTrue beta: {beta_true}")
print(f"Estimated beta: {beta_hat.tolist()}")
print(f"True intercept: {[1]}；Estimated intercept: {bias_hat.tolist()}")
print(f"Beta estimation error: {np.abs(beta_true - beta_hat).tolist()}")

def predict_x(u1, u2):
    with torch.no_grad():
        inputs = [u1, u2]  
        x_pred = model.layer['x'](inputs)
    return x_pred
    
def predict_y(u1, u2, u3):
    with torch.no_grad():
        x_pred = predict_x(u1, u2)
        inputs = [x_pred, u3] 
        y_pred = model.layer['y'](inputs)
    return y_pred

test_data = generate_true_data(n_samples=1000, beta=beta_true)
u1_test, u2_test, u3_test, x_true, y_true = (
    test_data[:, 0:1],
    test_data[:, 1:2],
    test_data[:, 2:3],
    test_data[:, 3:4],
    test_data[:, 4:5]
)

with torch.no_grad():
    x_pred = predict_x(u1_test, u2_test)
    y_pred = predict_y(u1_test, u2_test, u3_test)

mse_x = torch.mean((x_pred - x_true) ** 2).item()
mse_y = torch.mean((y_pred - y_true) ** 2).item()

print(f"\nMSE for x (on 1000 test samples): {mse_x:.4f}")
print(f"MSE for y (on 1000 test samples): {mse_y:.4f}")

Iter 1 | loss: 3.8409, s1: 4.7627, s2: 1.8434
Iter 1000 | loss: 1.6999, s1: 3.3690, s2: 3.3382
Iter 2000 | loss: 1.6798, s1: 3.3273, s2: 3.2951
Iter 3000 | loss: 1.6848, s1: 3.3049, s2: 3.2402
Iter 4000 | loss: 1.6571, s1: 3.3112, s2: 3.3083
Iter 5000 | loss: 1.6742, s1: 3.3247, s2: 3.3009
Iter 6000 | loss: 1.6634, s1: 3.3232, s2: 3.3197
Iter 7000 | loss: 1.6517, s1: 3.3083, s2: 3.3131
Iter 8000 | loss: 1.6812, s1: 3.3577, s2: 3.3531

True beta: [2, 1]
Estimated beta: [1.981688380241394, 1.0374611616134644]
True intercept: [1]；Estimated intercept: [1.02540922164917]
Beta estimation error: [0.018311619758605957, 0.037461161613464355]

MSE for x (on 1000 test samples): 0.0557
MSE for y (on 1000 test samples): 0.2202
